# 01 — Data Preparation
**Covers rubric steps 1–3:** describe the data, check data quality, document preparation.

**Input:** `data/raw/survey_results_public.csv` (untouched download)
**Output:** `data/processed/us_fulltime_clean.csv`

Run top to bottom. Every filtering or cleaning choice gets a markdown cell explaining *why*.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

## 1. Describe the data
Source, size, and what one row represents (one survey respondent).

In [ ]:
raw = pd.read_csv("../data/raw/survey_results_public.csv", low_memory=False)
print("Rows, columns:", raw.shape)
raw.head()

In [ ]:
# Variables we plan to use - verify every name against data/docs/survey_results_schema.csv
cols = ["ResponseId", "ConvertedCompYearly", "WorkExp", "YearsCode", "DevType",
        "RemoteWork", "Employment", "Country", "EdLevel", "OrgSize", "Age"]
missing_cols = [c for c in cols if c not in raw.columns]
print("Columns not found (check schema for the 2025 name):", missing_cols)

In [ ]:
# Data types and a peek at each variable
for c in [c for c in cols if c in raw.columns]:
    print(f"--- {c} ({raw[c].dtype}) ---")
    print(raw[c].value_counts(dropna=False).head(8), "\n")

**TODO:** Write the variable table here (name, meaning, type, units) for the report.

## 2. Define the population
Filter step by step and record the row count after each step. Check the exact country and employment labels in the output above before running.

In [ ]:
funnel = [("All respondents", len(raw))]

us = raw[raw["Country"] == "United States of America"]
funnel.append(("United States", len(us)))

ft = us[us["Employment"].str.contains("Employed, full-time", na=False)]
funnel.append(("Employed full-time", len(ft)))

paid = ft[ft["ConvertedCompYearly"].notna()].copy()
funnel.append(("Reported compensation", len(paid)))

pd.DataFrame(funnel, columns=["Step", "Rows remaining"])

## 3. Data quality
### 3a. Missing compensation — is it random?
Compare full-time U.S. respondents who reported pay with those who did not.

In [ ]:
ft = ft.copy()
ft["reported_pay"] = ft["ConvertedCompYearly"].notna()
print(ft["reported_pay"].value_counts(normalize=True))

for c in ["RemoteWork", "EdLevel"]:
    print(pd.crosstab(ft[c], ft["reported_pay"], normalize="columns").round(3), "\n")

**TODO:** Interpret — do non-reporters differ? If yes, note it as a limitation.

### 3b. Duplicates

In [ ]:
print("Duplicate ResponseIds:", paid["ResponseId"].duplicated().sum())
print("Fully identical rows:", paid.duplicated().sum())

### 3c. Extreme compensation values
Look before deciding. Pay is right-skewed, so use a log scale.

In [ ]:
print(paid["ConvertedCompYearly"].describe(percentiles=[.01, .05, .5, .95, .99]))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.log10(paid["ConvertedCompYearly"].clip(lower=1)), bins=60)
ax.set_xlabel("log10(Annual compensation, USD)")
ax.set_ylabel("Respondents")
ax.set_title("Reported compensation before outlier handling")
plt.show()

In [ ]:
# Outlier rule - CHOOSE ONE, justify it in markdown, and report how many rows it affects.
LOW, HIGH = 10_000, paid["ConvertedCompYearly"].quantile(0.99)
outside = ~paid["ConvertedCompYearly"].between(LOW, HIGH)
print(f"Rows outside [{LOW:,.0f}, {HIGH:,.0f}]: {outside.sum()} ({outside.mean():.1%})")

paid = paid[~outside].copy()
funnel.append(("After outlier rule", len(paid)))

**TODO:** Justify the rule (e.g., pay under $10k is implausible for full-time U.S. work; top 1% removed as likely entry errors or extreme cases).

## 4. Preparation
### 4a. Experience stored as text

In [ ]:
exp_map = {"Less than 1 year": 0.5, "More than 50 years": 51}
for c in ["WorkExp", "YearsCode"]:
    paid[c + "_num"] = pd.to_numeric(paid[c].replace(exp_map), errors="coerce")
    print(c, "unconvertible values:", paid[c + "_num"].isna().sum())

In [ ]:
bins = [0, 2, 5, 10, 20, np.inf]
labels = ["0-2", "3-5", "6-10", "11-20", "20+"]
paid["exp_band"] = pd.cut(paid["WorkExp_num"], bins=bins, labels=labels, include_lowest=True)
paid["exp_band"].value_counts().sort_index()

### 4b. Multi-select questions
Check whether `DevType` contains semicolon-separated answers, then choose and justify an approach.

In [ ]:
multi = paid["DevType"].str.contains(";", na=False)
print(f"DevType multi-select rows: {multi.sum()} ({multi.mean():.1%})")

In [ ]:
# Collapse rare roles (< 30 respondents) into "Other" so group medians are reliable
role_counts = paid["DevType"].value_counts()
rare = role_counts[role_counts < 30].index
paid["role"] = paid["DevType"].where(~paid["DevType"].isin(rare), "Other")
paid["role"].value_counts()

### 4c. Log compensation (for plots and models)

In [ ]:
paid["log_pay"] = np.log(paid["ConvertedCompYearly"])
paid = paid.rename(columns={"ConvertedCompYearly": "pay"})

## 5. Save the processed dataset

In [ ]:
keep = ["ResponseId", "pay", "log_pay", "WorkExp_num", "YearsCode_num", "exp_band",
        "role", "RemoteWork", "EdLevel", "OrgSize", "Age"]
paid[keep].to_csv("../data/processed/us_fulltime_clean.csv", index=False)
print("Saved:", paid[keep].shape)
pd.DataFrame(funnel, columns=["Step", "Rows remaining"])